In [1]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras
import time

from scikeras.wrappers import KerasRegressor
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import metrics

2026-02-05 18:55:33.694981: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-05 18:55:33.695528: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 18:55:34.039157: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 18:55:36.505499: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
# iniciando o tempo
start = time.time()
start

1770328537.9469364

In [3]:
# caminho para o dataset
path = '../archives/autos.csv'

# importando o dataset
df = pd.read_csv(path, encoding='ISO-8859-1')

# removendo as colunas desnecessárias
df = df.drop('dateCrawled', axis = 1)
df = df.drop('dateCreated', axis = 1)
df = df.drop('nrOfPictures', axis = 1)
df = df.drop('postalCode', axis = 1)
df = df.drop('lastSeen', axis = 1)
df = df.drop('name', axis = 1)
df = df.drop('seller', axis = 1)
df = df.drop('offerType', axis = 1)


In [4]:
# removendo os outliers
df = df[df.price > 10]
df = df.loc[df.price < 350000]

In [5]:
# dicionario para preencher os valores vazios com as modas (valores mais preenchidos para cada categoria)
values = {
    'vehicleType': 'limousine',
    'gearbox': 'manuell',
    'model': 'golf',
    'fuelType': 'benzin',
    'notRepairedDamage': 'nein'
}

# preenchendo os valores vazios
df = df.fillna(value=values)

In [6]:
# Separando os atributos em previsores e resposta
X = df.iloc[:, 1:12].values
y = df.iloc[:, 0].values

In [7]:
X, y

(array([['test', 'limousine', 1993, ..., 'benzin', 'volkswagen', 'nein'],
        ['test', 'coupe', 2011, ..., 'diesel', 'audi', 'ja'],
        ['test', 'suv', 2004, ..., 'diesel', 'jeep', 'nein'],
        ...,
        ['test', 'bus', 1996, ..., 'diesel', 'volkswagen', 'nein'],
        ['test', 'kombi', 2002, ..., 'diesel', 'volkswagen', 'nein'],
        ['control', 'limousine', 2013, ..., 'benzin', 'bmw', 'nein']],
       dtype=object),
 array([  480, 18300,  9800, ...,  9200,  3400, 28990]))

In [8]:
# transformando cada coluna do dataset em uma matriz binária
ohe = ColumnTransformer(transformers=[('OneHot', OneHotEncoder(), [0, 1, 3, 5, 8, 9, 10])], remainder='passthrough')

In [9]:
# aplicando a transformação
X = ohe.fit_transform(X).toarray()

In [10]:
# função para criar a rede neural
def create_network():
    k.clear_session()
    # criando a rede neural
    regressor = Sequential([
        tf.keras.layers.InputLayer(shape=(316,)),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=1, activation='linear')
    ])
    # compilando
    regressor.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mean_absolute_error'])
    return regressor

In [11]:
regressor = KerasRegressor(model=create_network, epochs=100, batch_size=300)

In [12]:
results = cross_val_score(estimator=regressor, X=X, y=y, cv=5, scoring='neg_mean_absolute_error')

2026-02-05 18:59:06.196578: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 3958.7249 - mean_absolute_error: 3958.7249
Epoch 2/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 3356.5801 - mean_absolute_error: 3356.5801
Epoch 3/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 3136.8027 - mean_absolute_error: 3136.8027
Epoch 4/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 3019.0105 - mean_absolute_error: 3019.0105
Epoch 5/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 2901.4402 - mean_absolute_error: 2901.4402
Epoch 6/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 2850.6147 - mean_absolute_error: 2850.6147
Epoch 7/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 2802.1997 - mean_absolute_error: 2802.2000
Epoch 8/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 2762.8169 - mean_absolute_error: 2762.8169
Epoch 9/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 2714.0471 - mean_absolute_error: 2714.0471
Epoch 10/100
959/959 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 26

In [ ]:
# final da execução
end = time.time()

In [ ]:
# tempo de execução
(end - start) / 60 / 60

1.1807400590843626

In [ ]:
# resultados
results

array([-2236.05358032, -2304.52222278, -2238.02792249, -2649.5216023 ,
       -2227.669701  ])

In [18]:
# média dos erros
abs(results.mean())

np.float64(2331.1590057780254)

In [19]:
# desvio padrao
results.std()

np.float64(161.55044902445954)